In [1]:
!pip install -r requirements.txt

Processing /wheels/flash_attn-2.6.3-cp310-cp310-linux_x86_64.whl (from -r requirements.txt (line 39))
flash_attn is already installed with the same version as the provided wheel. Use --force-reinstall to force an installation of the wheel.


# ПОДГОТОВКА

In [2]:
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import gc
from peft import PeftModel
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate, SystemMessagePromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
import dotenv
from langchain_openai import ChatOpenAI
import os

/usr/local/lib/python3.10/dist-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [3]:
dotenv.load_dotenv()

True

In [4]:
def clean_memory():
    for var in ['foundation_model', 'tokenizer']:
        if var in globals():
            del globals()[var]

    if torch.cuda.is_available():
        torch.cuda.ipc_collect()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.empty_cache()

    gc.collect()
    print('Memory is cleaned')

def print_memory():
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated(0) / 1024**3
        reserved = torch.cuda.memory_reserved(0) / 1024**3
        print(f'VRAM allocated {allocated}gb, reserved {reserved}gb')
    else:
        print('No cuda')


In [5]:
clean_memory()
print_memory()

Memory is cleaned
VRAM allocated 0.0gb, reserved 0.0gb


# Подготовка dataset для LLM as a judge через разметку двумя моделями

In [6]:
test_dataset = load_from_disk('test_dataset')

In [7]:
test_dataset

Dataset({
    features: ['conversations', 'source', 'score', 'openai_dialog', 'text'],
    num_rows: 198
})

In [8]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

In [9]:
model_name = 'mistralai/Mistral-7B-Instruct-v0.3'
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

# гарантируем eos_token_id
if tokenizer.eos_token_id is None and tokenizer.eos_token is not None:
    tokenizer.eos_token_id = tokenizer.convert_tokens_to_ids(tokenizer.eos_token)

foundation_model = AutoModelForCausalLM.from_pretrained(model_name,
                                                        quantization_config=bnb_config,
                                                        device_map="auto",
                                                        attn_implementation="flash_attention_2")

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [10]:
print_memory()

VRAM allocated 3.85457706451416gb, reserved 3.859375gb


In [11]:
def apply_model(model, row):
    chat = []
    for i in row['openai_dialog']:
        if i['role'] == 'user':
            chat.append(i)
            break

    prompt = tokenizer.apply_chat_template(
        chat,
        add_generation_prompt=True,
        tokenize=False,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    gen = model.generate(**inputs,
                         max_new_tokens=1024,
                         do_sample=False,
                         return_dict_in_generate=True,
                         repetition_penalty=1.5,
                         eos_token_id=tokenizer.eos_token_id,
                         pad_token_id=tokenizer.eos_token_id,)

    prompt_len = inputs["attention_mask"].sum(dim=1).item()
    new_tokens = gen.sequences[0, prompt_len:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=False)
    return answer
        

In [12]:
def apply_foundation_model(row):
    answer = apply_model(foundation_model, row)
    return {'foundation_model_answer': answer}

In [13]:
test_dataset = test_dataset.map(apply_foundation_model)

Map:   0%|          | 0/198 [00:00<?, ? examples/s]

From v4.47 onwards, when a model cache is to be returned, `generate` will return a `Cache` instance instead by default (as opposed to the legacy tuple of tuples format). If you want to keep returning the legacy format, please set `return_legacy_cache=True`.


In [14]:
lora_model = PeftModel.from_pretrained(foundation_model, './peft_lab_outputs/lora_adapter_3')
lora_model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): MistralForCausalLM(
      (model): MistralModel(
        (embed_tokens): Embedding(32768, 4096)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralFlashAttention2(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k

In [15]:
def apply_lora_model(row):
    answer = apply_model(lora_model, row)
    return {'lora_model_answer': answer}

In [16]:
test_dataset = test_dataset.map(apply_lora_model)

Map:   0%|          | 0/198 [00:00<?, ? examples/s]

In [26]:
print(test_dataset.select([3])['openai_dialog'])

Column([[{'content': 'How did the evolution of herbivorous mammals and their adaptations help them survive in their respective habitats and compete for resources with other animals?', 'role': 'user'}, {'content': 'The evolution of herbivorous mammals and their adaptations have played a significant role in their survival in various habitats and competition for resources with other animals. These adaptations can be observed in their anatomical, physiological, and behavioral traits, which have allowed them to exploit different food sources, avoid predation, and coexist with other species. Some of the key adaptations include:\n\n1. Dental adaptations: Herbivorous mammals have evolved specialized teeth for processing plant material. For example, many herbivores have sharp incisors for cutting and tearing plant material, and flat molars for grinding and breaking down fibrous plant matter. This allows them to efficiently consume and digest a wide variety of plant-based diets.\n\n2. Digestive 

In [31]:
test_dataset.save_to_disk('assessed_dataset2')

Saving the dataset (0/1 shards):   0%|          | 0/198 [00:00<?, ? examples/s]

# LLM-as-a-judge

In [50]:
class JudgeAnswer(BaseModel):
    chosen_model: int = Field(description='Model number, which is better. -1 for the first model, 1 for the second model. 0 if models are equal.')
    reason: str = Field(description='Reason why chosen model is better.')

In [51]:
llm = ChatOpenAI(
    api_key=os.environ['API_KEY'],
    base_url=os.environ['API_BASE_URL'],
    temperature=0.0,
    model='qwen-3-32b'
)

In [52]:
judge_llm = llm.with_structured_output(JudgeAnswer)  # важно

In [53]:
def get_prompt():
    return ChatPromptTemplate.from_messages(
        [
            SystemMessagePromptTemplate.from_template("""
                You are llm judge. You must compare two models.
                You are given request prompt between xml tags <instruction> and </instruction>
                You are given first model answer between xml tags <answer1> and </answer1>.
                You are given second model answer between xml tags <answer2> and </answer2>.
                You are given ideal answer between xml tags <ideal> and </ideal>

                Chose the best answer:
                - -1 if first model is better;
                - 0 if models are equeal;
                - 1 if second model is better;

                Criterias:
                - text style
                - correctness
                - faithfulness
                - precision
                - recall
            """),
            HumanMessagePromptTemplate.from_template("""
            Judge which model is better.
            Request was:
            <instruction>
            {instruction}
            <instruction>
            <answer1>
            {foundation_model_answer}
            </answer1>
            <answer2>
            {lora_model_answer}
            </answer2>
            <ideal>
            {ideal_answer}
            </ideal>
            """)
        ]
    )

In [54]:
def llm_judge(row):
    foundation_model_answer = row['foundation_model_answer']
    lora_model_answer = row['lora_model_answer']

    instruction = ''
    for i in row['openai_dialog']:
        if i['role'] == 'user':
            instruction = i['content']  
            break

    ideal_answer = ''
    for i in row['openai_dialog']:
        if i['role'] == 'assistant':
            ideal_answer = i['content']  
            break

    judge_answer = (get_prompt() | judge_llm).invoke({'foundation_model_answer': foundation_model_answer,
                                         'lora_model_answer': lora_model_answer,
                                         'instruction': instruction,
                                         'ideal_answer': ideal_answer,
                                         }, config={"callbacks": []})
    print('Chosen model: ', judge_answer.chosen_model, 'Reason: ', judge_answer.reason)
    return {
        'better_model': judge_answer.chosen_model,
        'choose_reason': judge_answer.reason,
    }

In [55]:
test_dataset = load_from_disk('assessed_dataset2')

In [56]:
test_dataset = test_dataset.map(llm_judge)

Map:   0%|          | 0/198 [00:00<?, ? examples/s]

Chosen model:  -1 Reason:  Answer1 is better than Answer2 because:

1. **Correctness**: Answer1 correctly describes key parasitic adaptations like crypticism (tapeworm larvae resembling plant seeds), immunomodulation by Toxoplasma gondii, evasion techniques, niche specialization, and coevolution. Answer2 contains several factual errors (e.g., 'Plasmidium falciparum' instead of 'Plasmodium falciparum', incorrect description of malaria lifecycle, confusing terminology). 

2. **Faithfulness**: Answer1 stays faithful to the scientific facts presented in the ideal answer and provides accurate examples. Answer2 strays from scientific accuracy.

3. **Precision**: Answer1 uses precise scientific terminology and specific examples (GRA-series antigens, Plasmodium spp., Trypanosoma cruzi, Giardia duodenalis) that align with the ideal answer. Answer2 lacks precision and contains misleading information.

4. **Recall**: Answer1 covers all major categories mentioned in the ideal answer (morphological

/usr/local/lib/python3.10/dist-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=JudgeAnswer(chosen_model=...ly captures correctly."), input_type=JudgeAnswer])
  return self.__pydantic_serializer__.to_python(


Chosen model:  1 Reason:  Answer 2 is significantly better than Answer 1. While Answer 1 provides some correct formulas for surface area and volume of a sphere, it contains several errors including incorrect volume formula (uses 4/3 * pi * r^3 but writes 'Volume(V)=(4/3)*pi*(Radius³)' with inconsistent formatting), incorrect example calculations, and confusing explanations. Answer 2 completely misses the point of the question, providing irrelevant information about diameter derivation and unrelated philosophical musings instead of addressing the surface area to volume ratio specifically. However, Answer 2 at least attempts to address the mathematical concept even if incorrectly, whereas Answer 1 provides partially correct information but with significant errors. The ideal answer shows the correct mathematical derivation showing the ratio is 3/r, which neither answer fully captures correctly.


/usr/local/lib/python3.10/dist-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=JudgeAnswer(chosen_model=... is marginally better."), input_type=JudgeAnswer])
  return self.__pydantic_serializer__.to_python(


...

/usr/local/lib/python3.10/dist-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=JudgeAnswer(chosen_model=...g challenge described."), input_type=JudgeAnswer])
  return self.__pydantic_serializer__.to_python(


Chosen model:  1 Reason:  Answer 2 is better because it provides more focused and practical strategies specifically addressing the core challenges mentioned in the question. While Answer 1 is comprehensive and well-structured, it includes some less relevant elements like peer collaborative workshops and regular assessments that don't directly address the core conceptual difficulties. Answer 2 focuses more effectively on the key areas: visual representation, real-world examples, practice exercises, and gamification techniques. It also better addresses the specific programming concepts mentioned (nesting, brackets, order of operations) with clearer connections to learning outcomes. The answer is more concise and directly applicable to the teaching challenge described.


/usr/local/lib/python3.10/dist-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=JudgeAnswer(chosen_model=...ion found in Answer 2."), input_type=JudgeAnswer])
  return self.__pydantic_serializer__.to_python(


Chosen model:  1 Reason:  Answer 2 is better because:

1. **Correctness**: Answer 2 correctly identifies that the cornea's primary function is refraction and protection, which aligns well with the ideal answer. While it mentions 'lens' instead of 'cornea' in the opening, it quickly corrects itself and provides accurate information about corneal functions.

2. **Faithfulness**: Answer 2 stays faithful to the core question about the cornea's function and structure, even though it initially misidentifies the subject as a 'lens'. It then provides comprehensive details about corneal functions including protection, refraction, and structural contributions.

3. **Precision**: Answer 2 gives precise details about the cornea's role in vision, mentioning its contribution to 65-75% of the eye's focusing power, which matches the ideal answer's precision.

4. **Recall**: Answer 2 covers all major aspects mentioned in the ideal answer: protection, refraction, transparency, curvature, and structural 

/usr/local/lib/python3.10/dist-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=JudgeAnswer(chosen_model=..., and well-documented."), input_type=JudgeAnswer])
  return self.__pydantic_serializer__.to_python(


In [57]:
test_dataset.save_to_disk('dataset_with_result')

Saving the dataset (0/1 shards):   0%|          | 0/198 [00:00<?, ? examples/s]

# Расчет результата

In [58]:
test_dataset = load_from_disk('dataset_with_result')

In [59]:
test_dataset

Dataset({
    features: ['conversations', 'source', 'score', 'openai_dialog', 'text', 'foundation_model_answer', 'lora_model_answer', 'better_model', 'choose_reason'],
    num_rows: 198
})

In [65]:
lora_better = 0
foundation_better = 0
model_equal = 0

def count_model_scores(row):
    global foundation_better
    global lora_better
    global model_equal
    if row['better_model'] == -1:
        foundation_better = foundation_better + 1
    elif row['better_model'] == 1:
        lora_better = lora_better + 1
    elif row['better_model'] == 0:
        model_equal = model_equal + 1

test_dataset.map(count_model_scores)

Map:   0%|          | 0/198 [00:00<?, ? examples/s]

Dataset({
    features: ['conversations', 'source', 'score', 'openai_dialog', 'text', 'foundation_model_answer', 'lora_model_answer', 'better_model', 'choose_reason'],
    num_rows: 198
})

In [67]:
print(f'Foundation model is better in {foundation_better}/{len(test_dataset)} cases')
print(f'Lora model is better in {lora_better}/{len(test_dataset)} cases')
print(f'Model are equal in {model_equal}/{len(test_dataset)} cases')

Foundation model is better in 20/198 cases
Lora model is better in 173/198 cases
Model are equal in 4/198 cases


In [68]:
final_score = (lora_better - foundation_better) / len(test_dataset)
print(f'Normalized lora score is {final_score}')

Normalized lora score is 0.7727272727272727


# Выводы
### Созданная LORA действительно улучшила качество следования инструкциям, что подтверждается методом оценки LLM-as-a-judge.
### В 173 из 198 случае LORA показала более высокое качество, чем базовая модель.